In [ ]:
import os
import arrow
import xarray as xr
import numpy as np
from pathlib import Path
import netCDF4 as nc

In [ ]:
SIM_DIR = '/ocean/atall/MOAD/Model/202410b/oxygen/' # 202410b, not the last version

out_dir = '/ocean/atall/MOAD/Model/202410b/station_netcdf_202410b'
os.makedirs(out_dir, exist_ok=True)


In [ ]:
# Coordinates of stations in lat/lon
stations = {
    'SAR003': (-122.49000, 48.10833),
    'PSS019': (-122.30000, 48.01167),
    'SKG003': (-122.48830, 48.29667),
    'HCB003': (-123.00830, 47.53833),
    'HCB004': (-123.02330, 47.35667),
    'ADM003': (-122.48180, 47.87917),
    'PSB003': (-122.44170, 47.66000),
    'SIN001': (-122.64170, 47.55000),
    'EAP001': (-122.38000, 47.41667),
    'CMB003': (-122.44830, 47.29000),
    'CSE001': (-122.84330, 47.26500),
    'GOR001': (-122.63330, 47.18333),
}


syear, smonth, sday = (2014, 1, 1)
eyear, emonth, eday = (2014, 12, 31)
startdate = arrow.get(syear, smonth, sday)
enddate   = arrow.get(eyear, emonth, eday)


In [ ]:
# Load grid and mask
grid_dir = Path("/ocean/atall/MOAD/grid/")
grid_map = Path("grid_from_lat_lon_mask999.nc")
grid_lons_lats = xr.open_dataset(grid_dir / grid_map)
meshmask0 = xr.open_dataset('/ocean/atall/MOAD/grid/mesh_mask202108.nc')
meshmask = xr.open_dataset('/ocean/atall/MOAD/grid/mesh_mask_202310b.nc')
tmask0 = meshmask0.tmask
mbathy0 = meshmask0.mbathy
tmask1 = meshmask.tmask
mbathy1 = meshmask.mbathy

mesh = nc.Dataset('/ocean/atall/MOAD/grid/mesh_mask_202310b.nc')
bathy = nc.Dataset('/ocean/atall/MOAD/grid/bathymetry_202310b.nc')
depth = mesh.variables['gdept_0'][:]

deptht = mesh['gdept_0'][0, :, 0, 0]

In [ ]:
# Coordinates of stations in grid indices
station_indices = {}

for name, (lon_sta, lat_sta) in stations.items():

    jj = grid_lons_lats.jj.sel(
        lats=lat_sta,
        lons=lon_sta,
        method='nearest'
    ).item()

    ii = grid_lons_lats.ii.sel(
        lats=lat_sta,
        lons=lon_sta,
        method='nearest'
    ).item()

    station_indices[name] = (jj, ii)


In [1]:
for station_name, (jj, ii) in station_indices.items():

    print(f'\nProcessing {station_name}')

    ds_daily = []

    for day_arrow in arrow.Arrow.range('day', startdate, enddate):

        year  = day_arrow.year
        yr2   = day_arrow.strftime("%y")
        month = day_arrow.month
        Month = day_arrow.strftime("%b").lower()
        day   = day_arrow.day

        base = (
            f'{SIM_DIR}{day:02}{Month}{yr2}/'
            f'SalishSea_1d_'
            f'{year}{month:02}{day:02}_'
            f'{year}{month:02}{day:02}'
        )

        fchem = f'{base}_chem_T.nc'
        fgrid = f'{base}_grid_T.nc'
        fbiol = f'{base}_biol_T.nc'

        
        vars_dict = {}

        # chem file
        with xr.open_dataset(fchem) as ds_chem:
            for var in ds_chem.data_vars:
                data = ds_chem[var]
                # skip non-4D variables
                if len(data.dims) < 4:
                    continue
                try:
                    vars_dict[var] = data[0, :, jj, ii].values
                except:
                    continue

        # grid file
        with xr.open_dataset(fgrid) as ds_grid:
            for var in ds_grid.data_vars:
                data = ds_grid[var]
                if len(data.dims) < 4:
                    continue
                try:
                    vars_dict[var] = data[0, :, jj, ii].values
                except:
                    continue

        # biol file
        if os.path.exists(fbiol):
            with xr.open_dataset(fbiol) as ds_biol:
                for var in ds_biol.data_vars:
                    data = ds_biol[var]
                    if len(data.dims) < 4:
                        continue
                    try:
                        vars_dict[var] = data[0, :, jj, ii].values
                    except:
                        continue

        # Build daily dataset for this station and day
        ds_day = xr.Dataset()
        for var, values in vars_dict.items():
            ds_day[var] = xr.DataArray(values, dims=['deptht'], coords={'deptht': deptht})

        # add time dimension
        ds_day = ds_day.expand_dims(time_counter=[np.datetime64(day_arrow.datetime)])
        ds_daily.append(ds_day)
        print(f'added {day_arrow.format("YYYY-MM-DD")}')

    # Concatenate daily datasets into one dataset for the station
    ds_station = xr.concat(ds_daily, dim='time_counter')

    # Add metadata
    ds_station.attrs['station'] = station_name
    ds_station.attrs['jj_index'] = jj
    ds_station.attrs['ii_index'] = ii

    ds_station.attrs['longitude'] = stations[station_name][0]
    ds_station.attrs['latitude'] = stations[station_name][1]

    # Save to netCDF
    outfile = os.path.join(out_dir, 
        f'{station_name}_2014.01.01_2014.12.31_ssc.nc'
    )

    ds_station.to_netcdf(outfile)

    print(f'\nSaved:')
    print(outfile)

/tmp/ipykernel_629574/2789231617.py:227: UserWarning: no explicit representation of timezones available for np.datetime64
  time_counter=[np.datetime64(day_arrow.datetime)]


  added 2014-04-19


/tmp/ipykernel_629574/2789231617.py:227: UserWarning: no explicit representation of timezones available for np.datetime64
  time_counter=[np.datetime64(day_arrow.datetime)]


  added 2014-04-20


/tmp/ipykernel_629574/2789231617.py:227: UserWarning: no explicit representation of timezones available for np.datetime64
  time_counter=[np.datetime64(day_arrow.datetime)]


...

  added 2014-12-29


/tmp/ipykernel_629574/2789231617.py:227: UserWarning: no explicit representation of timezones available for np.datetime64
  time_counter=[np.datetime64(day_arrow.datetime)]


  added 2014-12-30


/tmp/ipykernel_629574/2789231617.py:227: UserWarning: no explicit representation of timezones available for np.datetime64
  time_counter=[np.datetime64(day_arrow.datetime)]


  added 2014-12-31

Saved:
/ocean/atall/MOAD/Model/202410b/station_netcdf_202410b/GOR001_2014.01.01_2014.12.31_ssc.nc
